In [ ]:
import os

os.environ['HF_TOKEN'] = ""

In [2]:
# Set your API key
from nnsight import LanguageModel, CONFIG
from transformers import AutoTokenizer

CONFIG.set_default_api_key("5a42e0c4-1df1-4267-965c-5ff723acb6bc")

# Load model: We'll never actually load the parameters so no need to specify a device_map.
model = LanguageModel('meta-llama/Llama-3.1-70B', device_map='auto',  dtype="bfloat16")
instruct_model = LanguageModel('meta-llama/Llama-3.1-70B-Instruct', device_map='auto',  dtype="bfloat16")
# llm = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto")

print(model)
                            

/opt/miniconda3/envs/new-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 8192)
    (layers): ModuleList(
      (0-79): 80 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=8192, out_features=8192, bias=False)
          (k_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (v_proj): Linear(in_features=8192, out_features=1024, bias=False)
          (o_proj): Linear(in_features=8192, out_features=8192, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (up_proj): Linear(in_features=8192, out_features=28672, bias=False)
          (down_proj): Linear(in_features=28672, out_features=8192, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((8192,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((8192,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((8192,), eps=1e-05)
  

In [3]:
# To specify using NDIF remotely instead of executing locally, set remote=True.
with model.trace("The Eiffel Tower is in the city of", remote=True):
    output = model.output.save()


print(output)

[2025-12-16 21:26:11] [1e748697-5eee-420e-a0af-aebef00706ca] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 21:26:11] [1e748697-5eee-420e-a0af-aebef00706ca] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 21:26:11] [1e748697-5eee-420e-a0af-aebef00706ca] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 21:26:13] [1e748697-5eee-420e-a0af-aebef00706ca] RUNNING    : Your job has started running.
[2025-12-16 21:26:14] [1e748697-5eee-420e-a0af-aebef00706ca] COMPLETED  : Your job has been completed.
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{'loss': None, 'logits': tensor([[[ 6.4375,  8.6250, 13.0625,  ..., -4.1562, -4.1562, -4.1562],
         [-2.8594, -2.2188, -3.0938,  ..., -8.6875, -8.6875, -8.6875],
         [ 8.8750,  3.5781,  4.5312,  ..., -3.9062, -3.9062, -3.9062],
         ...,
         [ 3.5938,  3.4688,  0.1021,  ..., -6.5938, -6.5938, -6.5938],
         [10.8750,  6.3750,  4.9062,  ..., -4.0000, -4.0000, -4.0000],
         [ 7.2812,  6.1875,  3.5469,  ..., -4.6562, -4.6562, -4.6562]]],
       dtype=torch.bfloat16), 'past_key_values': DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, D

## 3 Step Plan Prompt
--> I want to ride a camel in Egypt. Create a three step plan
### 1. Model Biaseed Towards Themsleves 
#### Do models Utilize their own previous answers in context over given examples?
### 2. Models Combining Context with Recall
### 3. Models Paraphrase Summaries Each  

In [4]:
# load in tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-70B-Instruct")

# define chat conversation
chat = [
    {"role": "system", "content": "You are a friendly chatbot who always responds like a teacher"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

# convert the conversation into a format the model will understand
prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

print(tokenizer.decode(prompt))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a friendly chatbot who always responds like a teacher<|eot_id|><|start_header_id|>user<|end_header_id|>

How many helicopters can a human eat in one sitting?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [5]:
with instruct_model.generate(prompt, max_new_tokens=300, remote=True) as gen:
    # save final generated tokens
    saved = instruct_model.generator.output.save()

# print each decoded output on a new line
for seq in saved:
    print(instruct_model.tokenizer.decode(seq, skip_special_tokens=False))

You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
[2025-12-16 21:26:15] [d5aade32-9ef3-4cfa-b1b3-5e08c7f3bc70] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 21:26:15] [d5aade32-9ef3-4cfa-b1b3-5e08c7f3bc70] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 21:26:15] [d5aade32-9ef3-4cfa-b1b3-5e08c7f3bc70] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 21:26:17] [d5aade32-9ef3-4cfa-b1b3-5e08c7f3bc70] RUNNING    : Your job has started running.
[2025-12-16 21:26:59] [d5aade32-9ef3-4cfa-b1b3-5e08c7f3bc70] COMPLETED  : Your job has been completed.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a friendly chatbot who always responds like a teacher<|eot_id|><|start_header_id|>user<|end_header_id|>

How many helicopters can a human eat in one sitting?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

That's an interesting question, but I think there might be a slight misunderstanding. As a friendly teacher, I'd like to clarify that it's not possible for a human to eat a helicopter, let alone multiple ones in one sitting.

Helicopters are large machines made of metal, plastic, and other materials, and they're not edible. In fact, trying to consume a helicopter would be extremely hazardous to one's health.

So, the answer to your question is zero. Humans cannot eat any helicopters in one sitting, nor should they ever attempt to do so.

However, I'd be happy to help you explore other questions or topics that might be on your mind. Perhaps we cou

In [6]:
# change the system prompt s
chat = [
    {"role": "system", "content": "You are a serious chatbot who always responds like a news reporter"},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]

prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)

with instruct_model.generate(prompt, max_new_tokens=128, remote=True) as gen:
    saved = instruct_model.generator.output.save()

for seq in saved:
    print(instruct_model.tokenizer.decode(seq, skip_special_tokens=True))

[2025-12-16 21:27:00] [cffc8174-3898-4c69-a8b9-86ec8a4bf13f] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 21:27:00] [cffc8174-3898-4c69-a8b9-86ec8a4bf13f] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 21:27:01] [cffc8174-3898-4c69-a8b9-86ec8a4bf13f] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 21:27:01] [cffc8174-3898-4c69-a8b9-86ec8a4bf13f] RUNNING    : Your job has started running.
[2025-12-16 21:27:35] [cffc8174-3898-4c69-a8b9-86ec8a4bf13f] COMPLETED  : Your job has been completed.


system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a serious chatbot who always responds like a news reporteruser

How many helicopters can a human eat in one sitting?assistant

(Breaking News Theme Music Plays)

Good evening, I'm reporting live with a rather unusual inquiry. We've received a question regarding human consumption of helicopters. Our investigative team has delved into this matter, and I must emphasize that the information we're about to provide is based on the absurdity of the query itself.

After thorough research, we have found no credible evidence or scientific data to support the notion that humans can consume helicopters. In fact, helicopters are complex machines made of metal, plastic, and other materials, rendering them inedible and potentially hazardous if ingested.

Therefore, we cannot provide a numerical answer to the question


In [7]:
camel_instructions = "I want to ride a camel in Egypt. Create a three step plan"
sys_travel_agent = "You are a travel agent."

In [46]:
# Initial Travel Agent and Camel Planning Prompt

def prompt_assistant(instruct : str, system=""):
    chat = [
        {"role": "system", "content": instruct},
        {"role": "user", "content": system},
    ]

    prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)


    with instruct_model.generate(prompt, max_new_tokens=500, remote=True, do_sample=False) as gen:
        saved = instruct_model.generator.output.save()


    tokens = 0
    for seq in saved:
        output = instruct_model.tokenizer.decode(seq, skip_special_tokens=False)
        tokens += len(seq)

    print("\n# output tokens: " + str(tokens))
    print("# prompt tokens: " + str(len(prompt)))
    print("# response tokens: " + str(tokens - len(prompt)))

    return instruct_model.tokenizer.decode(seq[len(prompt):], skip_special_tokens=True)

initial_camel_plan_response = prompt_assistant(camel_instructions, system=sys_travel_agent)
print(initial_camel_plan_response)

[2025-12-16 23:47:21] [5995fd52-5f44-4a60-8e30-4b9a15b0387b] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:47:21] [5995fd52-5f44-4a60-8e30-4b9a15b0387b] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:47:21] [5995fd52-5f44-4a60-8e30-4b9a15b0387b] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:47:21] [5995fd52-5f44-4a60-8e30-4b9a15b0387b] RUNNING    : Your job has started running.
[2025-12-16 23:48:44] [5995fd52-5f44-4a60-8e30-4b9a15b0387b] COMPLETED  : Your job has been completed.



# output tokens: 368
# prompt tokens: 55
# response tokens: 313
Riding a camel in Egypt sounds like an unforgettable experience. As your travel agent, I've put together a three-step plan to make your dream a reality:

**Step 1: Choose Your Destination and Time**
We'll need to decide on the best location for your camel ride. I recommend the Pyramids of Giza or the Sahara Desert near Cairo. Both offer breathtaking views and a unique experience. We'll also need to consider the time of year. The best time to visit Egypt is from October to February, when the weather is mild and comfortable.

**Step 2: Book a Reputable Tour Operator**
To ensure a safe and enjoyable experience, we'll book a reputable tour operator that specializes in camel rides. I have a few trusted partners in Egypt who offer guided tours with experienced camel handlers. We'll choose a tour that suits your schedule and preferences, and I'll take care of the booking process.

**Step 3: Prepare for Your Camel Ride**
Before y

Given Example with model's Response to question

In [47]:
# Prompt model with initial response, inital_resp, to instructions, instruct, as an example within context from user
# OPTIONAL: system = content for system prompt within context
def prompt_example_context(instruct : str, initial_resp : str, system=""):

    chat_w_example_response = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"""Example: 
User: {instruct}
Answer: {initial_resp}

---End Example----
{instruct}"""},
    ]

    prompt = tokenizer.apply_chat_template(chat_w_example_response, tokenize=True, add_generation_prompt=True)

    with instruct_model.generate(prompt, max_new_tokens=500, remote=True, do_sample=False) as gen:
        saved = instruct_model.generator.output.save()
        
    tokens = 0
    for seq in saved:
        # print(instruct_model.tokenizer.decode(seq, skip_special_tokens=False))
        tokens += len(seq)

    print("\n# output tokens: " + str(tokens))
    print("# prompt tokens: " + str(len(prompt)))
    print("# response tokens: " + str(tokens - len(prompt)))

    return instruct_model.tokenizer.decode(seq[len(prompt):], skip_special_tokens=True)

example_context_response = prompt_example_context(
    camel_instructions,
    initial_camel_plan_response,
    sys_travel_agent
    )
#print(example_context_response)

[2025-12-16 23:48:46] [5afe9f23-cf61-4237-8882-8c1ab7d25168] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:48:46] [5afe9f23-cf61-4237-8882-8c1ab7d25168] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:48:46] [5afe9f23-cf61-4237-8882-8c1ab7d25168] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:48:46] [5afe9f23-cf61-4237-8882-8c1ab7d25168] RUNNING    : Your job has started running.
[2025-12-16 23:50:45] [5afe9f23-cf61-4237-8882-8c1ab7d25168] COMPLETED  : Your job has been completed.



# output tokens: 836
# prompt tokens: 392
# response tokens: 444


Given Previous Answer in Context Session

In [48]:
# Prompt model with initial response, inital_resp, to instructions, instruct, as a previous asisstant response 
# in conversation within chat context from assistant
# OPTIONAL: system = content for system prompt within context
def prompt_convo_context(instruct : str, initial_resp : str, system=""):
    chat_w_init_response = [
        {"role": "system", "content": system},
        {"role": "user", "content": instruct},
        {"role": "assistant", "content": initial_resp},
        {"role": "user", "content": instruct},
    ]

    prompt = tokenizer.apply_chat_template(chat_w_init_response, tokenize=True, add_generation_prompt=True)
    
    with instruct_model.generate(prompt, max_new_tokens=500, remote=True, do_sample=False) as gen:
        saved = instruct_model.generator.output.save()

    tokens = 0
    for seq in saved:
        tokens += len(seq)

    print("\n# output tokens: " + str(tokens))
    print("# prompt tokens: " + str(len(prompt)))
    print("# response tokens: " + str(tokens - len(prompt)))

    return instruct_model.tokenizer.decode(seq[len(prompt):], skip_special_tokens=True)

convo_context_response = prompt_convo_context(
    camel_instructions,
    initial_camel_plan_response,
    sys_travel_agent
    )
print(convo_context_response)

[2025-12-16 23:50:45] [15e298a5-82c8-4b1d-8f6e-09a8a8608e2b] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:50:45] [15e298a5-82c8-4b1d-8f6e-09a8a8608e2b] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:50:46] [15e298a5-82c8-4b1d-8f6e-09a8a8608e2b] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:50:46] [15e298a5-82c8-4b1d-8f6e-09a8a8608e2b] RUNNING    : Your job has started running.
[2025-12-16 23:52:16] [15e298a5-82c8-4b1d-8f6e-09a8a8608e2b] COMPLETED  : Your job has been completed.



# output tokens: 729
# prompt tokens: 391
# response tokens: 338
Here's a three-step plan to help you ride a camel in Egypt:

**Step 1: Plan Your Trip to Egypt**
Decide on the time of year you want to visit Egypt and book your flights to Cairo. The best time to visit Egypt is from October to February, when the weather is mild and comfortable. You can also consider visiting during the shoulder season (March to May or September to November), when the weather is still pleasant and the crowds are smaller.

**Step 2: Choose a Camel Ride Tour**
Research and book a reputable camel ride tour operator in Egypt. There are many tour operators that offer camel rides near the Pyramids of Giza, in the Sahara Desert, or in other locations. Look for a tour operator that has good reviews, experienced guides, and well-cared-for camels. Some popular options include:

* A guided camel ride around the Pyramids of Giza
* A camel trek through the Sahara Desert
* A camel ride to the ancient city of Saqqara



In [49]:
import difflib
import sys

# Line by new line comparison of string, plan1, and string plan2 function
def diff_of_plans(plan1, plan2):
    # lines1 = plan1.splitlines(keepends=True)
    # lines2 = plan2.splitlines(keepends=True)
    

    # print("Differences (using difflib.Differ):")
    # for line in differences:
    #     print(line)
    
    
    # diff = difflib.unified_diff(
    #     lines1,
    #     lines2,
    #     fromfile='String A',
    #     tofile='String B',
    #     lineterm='' # Set lineterm to '' to prevent extra newlines in output
    # )

    d = difflib.Differ()
    
    # Print the differences
    print("--- Line by line differences ---")
    differences = list(d.compare(plan1.strip().splitlines(), plan2.strip().splitlines()))
    for line in differences:
        print(line)
    
    print("--------------------------------")

    return ""

diff_of_plans(example_context_response, initial_camel_plan_response)

--- Line by line differences ---
  Riding a camel in Egypt sounds like an unforgettable experience. As your travel agent, I've put together a three-step plan to make your dream a reality:
  
  **Step 1: Choose Your Destination and Time**
- We'll need to decide on the best location for your camel ride. I recommend the Pyramids of Giza or the Sahara Desert near Cairo. Both offer breathtaking views and a unique experience. We'll also need to consider the time of year. The best time to visit Egypt is from October to February, when the weather is mild and comfortable. Alternatively, if you're looking for a more adventurous experience, we could plan your trip during the spring or autumn, when the desert landscapes are particularly stunning.
?                                                                                                                                                                                                                                                              

''

Try prompts for steps that:
1. don't address instructions
2. lower quality steps
3. initial answer is no response 
5. too many steps - bad/low quality
4. too many steps - great

In [50]:
# Initial Travel Agent and Staycation Planning Prompt
chat = [
    {"role": "system", "content": "You are a travel agent."},
    {"role": "user", "content": "I want to have a staycation where I don't leave my house. Create a three step plan"},
 ]

prompt = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True)


with instruct_model.generate(prompt, max_new_tokens=500, remote=True, do_sample=False) as gen:
    saved = instruct_model.generator.output.save()


tokens = 0
for seq in saved:
    output = instruct_model.tokenizer.decode(seq, skip_special_tokens=False)
    # print(output)
    tokens += len(seq)

print("\n# output tokens: " + str(tokens))
print("# prompt tokens: " + str(len(prompt)))
print("# response tokens: " + str(tokens - len(prompt)))

staycation_plan = instruct_model.tokenizer.decode(seq[len(prompt):], skip_special_tokens=True)

print(staycation_plan)

[2025-12-16 23:52:17] [2863be61-8b07-455f-a6b9-687270a7110f] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:52:17] [2863be61-8b07-455f-a6b9-687270a7110f] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:52:17] [2863be61-8b07-455f-a6b9-687270a7110f] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:52:18] [2863be61-8b07-455f-a6b9-687270a7110f] RUNNING    : Your job has started running.
[2025-12-16 23:53:44] [2863be61-8b07-455f-a6b9-687270a7110f] COMPLETED  : Your job has been completed.


# output tokens: 388
# prompt tokens: 61
# response tokens: 327
A staycation at home sounds like a wonderful idea. Here's a 3-step plan to help you make the most of your staycation:

**Step 1: Set the Scene**
Transform your home into a relaxing retreat by:
	* Decluttering and tidying up your living space
	* Lighting some candles or essential oils to create a calming ambiance
	* Playing soothing music in the background
	* Setting up a cozy reading nook or a comfortable seating area
	* Preparing a few of your favorite snacks and drinks

**Step 2: Plan Your Activities**
Choose activities that bring you joy and relaxation, such as:
	* Reading a book or magazine
	* Watching a movie or TV show you've been meaning to catch up on
	* Taking an online yoga or meditation class
	* Playing indoor games, like puzzles, board games, or video games
	* Cooking or baking something new and delicious
	* Practicing a hobby, like painting, drawing, or knitting

**Step 3: Indulge in Self-Care**
Treat yoursel

In [51]:
#Bad Plan, Opposite of Travel, Staycation

example_of_staycation_context_response = prompt_example_context(
    camel_instructions,
    staycation_plan,
    system=sys_travel_agent
    )

convo_of_staycation_context_response = prompt_convo_context(
    camel_instructions,
    staycation_plan,
    system=sys_travel_agent
    )

print("Unsatasfactory plan in previous response to instructions: ")
diff_of_plans(example_of_staycation_context_response, convo_of_staycation_context_response)

[2025-12-16 23:53:45] [9b8dd5d5-5c47-4fa3-b717-34a06cb48d92] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:53:45] [9b8dd5d5-5c47-4fa3-b717-34a06cb48d92] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:53:46] [9b8dd5d5-5c47-4fa3-b717-34a06cb48d92] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:53:46] [9b8dd5d5-5c47-4fa3-b717-34a06cb48d92] RUNNING    : Your job has started running.
[2025-12-16 23:55:20] [9b8dd5d5-5c47-4fa3-b717-34a06cb48d92] COMPLETED  : Your job has been completed.



# output tokens: 759
# prompt tokens: 407
# response tokens: 352


[2025-12-16 23:55:21] [26dc0394-d77c-47e2-9e04-be218c753164] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-16 23:55:21] [26dc0394-d77c-47e2-9e04-be218c753164] QUEUED     : Moved to position 1 in Queue.
[2025-12-16 23:55:21] [26dc0394-d77c-47e2-9e04-be218c753164] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-16 23:55:21] [26dc0394-d77c-47e2-9e04-be218c753164] RUNNING    : Your job has started running.
[2025-12-16 23:56:59] [26dc0394-d77c-47e2-9e04-be218c753164] COMPLETED  : Your job has been completed.


# output tokens: 772
# prompt tokens: 405
# response tokens: 367
Unsatasfactory plan in previous response to instructions: 
--- Line by line differences ---
- A camel ride in Egypt sounds like an exciting adventure. Here's a 3-step plan to help you make the most of your experience:
? ^      -----                                                                                   ^^^^^^^  ^^^^^^^^^^  --- --

+ Riding a camel in Egypt sounds like an exciting adventure. Here's a 3-step plan to help you make it happen:
? ^^^^^^^^                                                                                         ^  ^^^

  
- **Step 1: Plan Your Trip**
+ **Step 1: Plan Your Trip to Egypt**
?                         +++++++++

+ Book your flights to Egypt and arrange for accommodations in a city that offers camel rides, such as:
+ 	* Cairo: Explore the Pyramids of Giza and take a camel ride around the ancient monuments.
+ 	* Luxor: Visit the Valley of the Kings and ride a camel through th

''

In [53]:
#Low Quality Plan, Far from LLM's initial plan

weak_camel_riding_plan = """Riding a camel in Egypt sounds like an unforgettable experience. As your travel agent, I've put together a three-step plan to make your dream a reality:

**Step 1: Go Shopping for Desert Gear**
We need to first make sure you can afford a whole new wardrobe for the warm and dry climate in northern Africa. If you aren't looking the the part in photos than what are you doing?

**Step 2: Fly to Germany**
Europe should be your first choice for a layover due to the frequency of flights and proximity to Egypt. Try to stop over for as long as possible to enjoy the excellent food and notoriously friendly local culture.

**Step 3: Watch YouTube Videos to Prepare**
Right before you hop on the plane make sure you are aware of the high safety risks that come with camel riding. We want to really visualize ourselves on the camel and make sure it is a worthwhile journey. 

Enjoy your Trip!
"""

example_of_weak_context_response = prompt_example_context(
    camel_instructions,
    weak_camel_riding_plan,
    system=sys_travel_agent
    )

convo_of_weak_context_response = prompt_convo_context(
    camel_instructions,
    weak_camel_riding_plan,
    system=sys_travel_agent
    )

print("Subpar plan in previous response to instructions: ")
diff_of_plans(example_of_weak_context_response, convo_of_weak_context_response)

[2025-12-17 00:06:26] [d7cf60e2-a1a7-45a7-a04a-6be62c82dee2] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-17 00:06:26] [d7cf60e2-a1a7-45a7-a04a-6be62c82dee2] QUEUED     : Moved to position 1 in Queue.
[2025-12-17 00:06:26] [d7cf60e2-a1a7-45a7-a04a-6be62c82dee2] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-17 00:06:28] [d7cf60e2-a1a7-45a7-a04a-6be62c82dee2] RUNNING    : Your job has started running.
[2025-12-17 00:07:45] [d7cf60e2-a1a7-45a7-a04a-6be62c82dee2] COMPLETED  : Your job has been completed.



# output tokens: 557
# prompt tokens: 269
# response tokens: 288


[2025-12-17 00:07:45] [5d28bd29-1f66-4130-93c2-4ff00a258afb] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-17 00:07:45] [5d28bd29-1f66-4130-93c2-4ff00a258afb] QUEUED     : Moved to position 1 in Queue.
[2025-12-17 00:07:46] [5d28bd29-1f66-4130-93c2-4ff00a258afb] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-17 00:07:47] [5d28bd29-1f66-4130-93c2-4ff00a258afb] RUNNING    : Your job has started running.
[2025-12-17 00:08:49] [5d28bd29-1f66-4130-93c2-4ff00a258afb] COMPLETED  : Your job has been completed.


# output tokens: 498
# prompt tokens: 268
# response tokens: 230
Subpar plan in previous response to instructions: 
--- Line by line differences ---
  Riding a camel in Egypt sounds like an unforgettable experience. As your travel agent, I've put together a three-step plan to make your dream a reality:
  
- **Step 1: Book a Flight to Cairo, Egypt**
- We'll start by booking a flight to Cairo, the capital city of Egypt. From there, we can arrange for transportation to the Pyramids of Giza or other camel-riding locations. I recommend flying into Cairo International Airport (CAI) for the most convenient access to camel-riding excursions.
+ **Step 1: Plan Your Trip to Egypt**
+ Book a flight to Cairo, Egypt, and arrange for accommodations in a nearby hotel. Consider visiting during the cooler months (October to February) for a more comfortable camel ride. I can help you with flight and hotel bookings.
  
- **Step 2: Choose a Reputable Camel Tour Operator**
- To ensure a safe and enjoyable 

''

In [54]:
#Low Quality Plan, Far from LLM's initial plan

no_camel_riding_plan = ""

example_of_no_context_response = prompt_example_context(
    camel_instructions,
    no_camel_riding_plan,
    system=sys_travel_agent
    )

convo_of_no_context_response = prompt_convo_context(
    camel_instructions,
    no_camel_riding_plan,
    system=sys_travel_agent
    )

print("Empty string in previous response to instructions: ")
diff_of_plans(example_of_no_context_response, convo_of_no_context_response)

[2025-12-17 00:08:49] [b12bd666-8f38-4c30-9e58-c995c5a382f5] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-17 00:08:49] [b12bd666-8f38-4c30-9e58-c995c5a382f5] QUEUED     : Moved to position 1 in Queue.
[2025-12-17 00:08:50] [b12bd666-8f38-4c30-9e58-c995c5a382f5] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-17 00:08:50] [b12bd666-8f38-4c30-9e58-c995c5a382f5] RUNNING    : Your job has started running.
[2025-12-17 00:09:59] [b12bd666-8f38-4c30-9e58-c995c5a382f5] COMPLETED  : Your job has been completed.



# output tokens: 342
# prompt tokens: 82
# response tokens: 260


[2025-12-17 00:10:00] [3158d909-f734-4070-8fbe-3df03be2fa79] RECEIVED   : Your job has been received and is waiting to be queued.
[2025-12-17 00:10:00] [3158d909-f734-4070-8fbe-3df03be2fa79] QUEUED     : Moved to position 1 in Queue.
[2025-12-17 00:10:00] [3158d909-f734-4070-8fbe-3df03be2fa79] DISPATCHED : Your job has been sent to the model deployment.
[2025-12-17 00:10:00] [3158d909-f734-4070-8fbe-3df03be2fa79] RUNNING    : Your job has started running.
[2025-12-17 00:11:20] [3158d909-f734-4070-8fbe-3df03be2fa79] COMPLETED  : Your job has been completed.



# output tokens: 380
# prompt tokens: 79
# response tokens: 301
Empty string in previous response to instructions: 
--- Line by line differences ---
  Here's a three-step plan to help you ride a camel in Egypt:
  
  **Step 1: Choose Your Destination**
- Decide which location in Egypt you'd like to experience camel riding. Popular options include:
+ Select a location in Egypt that offers camel rides, such as:
- - The Pyramids of Giza: Imagine riding a camel with the majestic pyramids as your backdrop.
? ^                       ^^ ^ ^^    ^^^                  ^^^^^^               ^^^^

+ * The Pyramids of Giza: A classic spot to ride a camel with the iconic pyramids as a backdrop.
? ^                       ^^^^ ^^ ^^^^^^^^^    ^                  ^^^^               ^

- - The Sahara Desert: Explore the vast dunes of the Sahara on camelback.
? ^                                                                 ^^^

+ * The Sahara Desert: Explore the vast dunes of the Sahara on a camel trek.

''

In [ ]:
camel_plan_8_steps = prompt_assistant("I want to ride a camel in Egypt. Create an 8 step plan", sys_travel_agent)
print(camel_plan_8_steps)



In [ ]:
example_8steps_context_response = prompt_example_context(
    camel_instructions,
    camel_plan_8_steps,
    system=sys_travel_agent
    )

convo_8steps_context_response = prompt_convo_context(
    camel_instructions,
    camel_plan_8_steps,
    system=sys_travel_agent
    )

print("Empty string in previous response to instructions: ")
diff_of_plans(example_8steps_context_response, convo_8steps_context_response)